# Comparación de Decision Tree y Random Forest

Este notebook compara el árbol de decisión desarrollado por el equipo con
Random Forest de scikit-learn usando Heart Disease de UCI.

Ejecutá todas las celdas de arriba hacia abajo. Al final, ajustá los controles
y pulsá **Comparar modelos**. Las cinco particiones estratificadas se
comparten entre los modelos y usan semilla 42 para repetir una corrida.

El objetivo es binario: 0 conserva la clase 0 del dataset y 1 agrupa las
clases originales 1 a 4. Las métricas resumen las predicciones de
prueba de las cinco particiones juntas.


In [ ]:
import importlib.util
import subprocess
import sys

paquetes = {
    "ucimlrepo": "ucimlrepo",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "ipywidgets": "ipywidgets",
}
faltantes = [paquete for modulo, paquete in paquetes.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass


In [ ]:
from math import log2

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from ucimlrepo import fetch_ucirepo


## 1. Datos de Cleveland

Las columnas numéricas conservan su escala. Las columnas categóricas
incluyen códigos como sexo, tipo de dolor y resultado de estudios.
El árbol propio recibe los valores faltantes; el preprocesamiento de
Random Forest se ajusta solo con los datos de entrenamiento de cada partición.


In [ ]:
heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features.copy()
y = (heart_disease.data.targets.iloc[:, 0] > 0).astype(int)

columnas_numericas = ["age", "trestbps", "chol", "thalach", "oldpeak"]
columnas_categoricas = [columna for columna in X.columns if columna not in columnas_numericas]
indices_continuos = [X.columns.get_loc(columna) for columna in columnas_numericas]
atributos = list(range(X.shape[1]))

print(f"Filas: {len(X)} | Atributos: {X.shape[1]}")
print("Clases binarias:", y.value_counts().sort_index().to_dict())
print("Faltantes por columna:")
display(X.isna().sum().loc[lambda valores: valores > 0].to_frame("faltantes"))


## 2. Decision Tree propio

Se reutilizan las funciones de entrenamiento del árbol enriquecido.
Los atributos continuos se dividen por umbrales y los faltantes se
distribuyen entre ramas. La poda compara el error estimado del subárbol
con alternativas más simples.

El parámetro de criterio selecciona Information Gain o Gain Ratio.
La ganancia mínima sigue aplicándose a la ganancia de información
ponderada por el peso del nodo, incluso cuando se usa Gain Ratio.


In [ ]:
from math import expm1, isfinite, isnan, log, sqrt


def es_desconocido(value):
    """Reconoce None, "?" y NaN como valores desconocidos.
    Fuente conceptual: NUL, libro pp. 27–30; PDF pp. 1–4, §§3.1–3.2.
    El libro usa "?"; admitir None y NaN es una decisión de implementación.
    El nombre del auxiliar no aparece como función en el PDF.
    """
    return (value is None or (isinstance(value, str) and value == "?")
            or (isinstance(value, float) and isnan(value)))


def pesos_casos(rows, weights=None):
    """Prepara pesos separados, finitos y no negativos; por defecto usa uno por caso.
    Fuente conceptual: NUL, libro p. 29; PDF p. 3, §3.1.2.
    El concepto de peso procede del texto; la validación y el nombre son propios.
    """
    result = [1.0] * len(rows) if weights is None else [float(w) for w in weights]
    if len(result) != len(rows) or any(not isfinite(w) or w < 0 for w in result):
        raise ValueError("Provide one finite nonnegative weight per row.")
    return result


def frecuencias_clase(rows, weights=None):
    """Suma los pesos por clase sin modificar las filas ni sus pesos.
    Fuente conceptual: NUL, libro pp. 29–31; PDF pp. 3–5, §§3.1.2–3.2.
    Generaliza los conteos a casos fraccionarios; el nombre del auxiliar es propio.
    """
    counts = {}
    for row, weight in zip(rows, pesos_casos(rows, weights)):
        if weight > 0:
            counts[row[-1]] = counts.get(row[-1], 0.0) + weight
    return counts


def probabilidades_clase(rows, weights=None):
    """Normaliza las frecuencias ponderadas de las clases.
    Fuente conceptual: NUL, libro pp. 30–32; PDF pp. 4–6, §§3.1.3–3.2.
    El nombre es propio; devolver un diccionario vacío si no hay peso es una convención.
    """
    counts = frecuencias_clase(rows, weights)
    total = sum(counts.values())
    return {label: count / total for label, count in counts.items()}


def info(rows, weights=None):
    """Calcula info(T), la entropía de clases, admitiendo pesos por caso.
    Fuente conceptual: NUL, libro pp. 28–30; PDF pp. 2–4, §§3.1.1–3.2.
    info es la notación del libro; el argumento opcional de pesos es su adaptación a Python.
    """
    return -sum(p * log2(p) for p in probabilidades_clase(rows, weights).values())


def separar_casos_conocidos(rows, attribute, weights=None, threshold=None):
    """Separa las ramas conocidas de los casos con resultado desconocido.
    Fuente conceptual: NUL, libro pp. 28–29; PDF pp. 2–3, §§3.1.1–3.1.2.
    Devuelve grupos de filas/pesos y pares desconocidos; el nombre y formato son propios.
    """
    groups, unknown = {}, []
    for row, weight in zip(rows, pesos_casos(rows, weights)):
        if weight == 0:
            continue
        value = row[attribute]
        if es_desconocido(value):
            unknown.append((row, weight))
        else:
            branch = value if threshold is None else value <= threshold
            group, group_weights = groups.setdefault(branch, ([], []))
            group.append(row)
            group_weights.append(weight)
    return groups, unknown


def evaluar_prueba(rows, attribute, weights=None, threshold=None):
    """Calcula gain(X), split info(X) y su cociente para una prueba.
    Fuente conceptual: NUL, libro pp. 28–31; PDF pp. 2–5, §§3.1.1–3.2.
    El nombre agregado es propio: las expresiones del libro son campos del resultado.
    Una prueba sin dos ramas conocidas utilizables devuelve None por convención local.
    """
    groups, unknown = separar_casos_conocidos(rows, attribute, weights, threshold)
    masses = {branch: sum(w) for branch, (_, w) in groups.items()}
    known_weight = sum(masses.values())
    total = known_weight + sum(w for _, w in unknown)
    if len(groups) < 2 or known_weight == 0:
        return None
    known_rows = [row for group, _ in groups.values() for row in group]
    known_weights = [w for _, group_weights in groups.values() for w in group_weights]
    after = sum(masses[branch] / known_weight * info(group, group_weights)
                for branch, (group, group_weights) in groups.items())
    gain = known_weight / total * (info(known_rows, known_weights) - after)
    proportions = [mass / total for mass in masses.values()]
    if total > known_weight:
        proportions.append((total - known_weight) / total)
    split_info = -sum(p * log2(p) for p in proportions)
    return {"threshold": threshold, "gain": gain, "split_info": split_info,
            "gain_ratio": gain / split_info, "branch_weights": masses,
            "branch_probabilities": {branch: mass / known_weight
                                     for branch, mass in masses.items()}}


def particionar_con_desconocidos(rows, attribute, weights=None, threshold=None):
    """Reparte cada caso desconocido con peso w por la probabilidad de cada rama.
    Fuente conceptual: NUL, libro pp. 29–31; PDF pp. 3–5, §§3.1.2–3.2.
    No crea una rama para nulos. El nombre y el formato (grupos, probabilidades) son propios.
    """
    groups, unknown = separar_casos_conocidos(rows, attribute, weights, threshold)
    known_weight = sum(sum(w) for _, w in groups.values())
    if known_weight == 0:
        return {}, {}
    probabilities = {branch: sum(w) / known_weight for branch, (_, w) in groups.items()}
    for branch, (group, group_weights) in groups.items():
        for row, weight in unknown:
            group.append(row)
            group_weights.append(weight * probabilities[branch])
    return groups, probabilities


def evaluar_umbrales_continuos(rows, attribute, weights=None, reference_values=None,
                      min_branch_weight=2.0):
    """Devuelve las estadísticas de cortes binarios válidos, ordenados por umbral.
    Fuente conceptual: CON, libro pp. 25–26; PDF pp. 1–2, §2.4.
    El texto describe el procedimiento, no este nombre de función.
    Usa valores observados de referencia y peso conocido mínimo; no elige ni entrena.
    """
    if not isfinite(min_branch_weight) or min_branch_weight <= 0:
        raise ValueError("min_branch_weight must be finite and positive.")
    weights = pesos_casos(rows, weights)
    values = {row[attribute] for row, w in zip(rows, weights)
              if w > 0 and not es_desconocido(row[attribute])}
    reference = values if reference_values is None else {
        value for value in reference_values if not es_desconocido(value)}
    if any(not isinstance(v, (int, float)) or not isfinite(v) for v in values | reference):
        raise ValueError("Continuous values must be finite numbers.")
    if not values <= reference:
        raise ValueError("reference_values must include every known positive-weight value.")
    ordered = sorted(values)
    candidates = []
    for left, right in zip(ordered, ordered[1:]):
        midpoint = left / 2 + right / 2
        threshold = max((v for v in reference if v < right and v <= midpoint), default=left)
        stats = evaluar_prueba(rows, attribute, weights, threshold)
        if stats is not None and min(stats["branch_weights"].values()) >= min_branch_weight:
            candidates.append(stats)
    return candidates


def combinar_probabilidades_clase(distributions, branch_probabilities):
    """Combina distribuciones de clases con las probabilidades de sus ramas.
    Fuente conceptual: NUL, libro pp. 30–33; PDF pp. 4–7, §§3.1.3–3.3.
    El nombre es propio. Requiere distribuciones normalizadas y claves de ramas coincidentes.
    """
    combined = {}
    for branch, branch_probability in branch_probabilities.items():
        for label, probability in distributions[branch].items():
            combined[label] = combined.get(label, 0.0) + branch_probability * probability
    return combined


def errores_estimados(total_weight, observed_errors, confidence=0.25):
    """Aproxima el conteo de errores N por U_CF(E,N), no la tasa U_CF sola.
    Fuente conceptual: POD, libro pp. 40–42; PDF pp. 4–6, §§4.2–4.3.
    El nombre, tabla de coeficientes y rutina numérica no se transcriben del extracto.
    La aproximación existente interpola errores menores que uno entre cero y min(1,N).
    El ancla N para hojas de peso menor que uno es una extensión propia pendiente de aprobación.
    """
    n, e = total_weight, observed_errors
    if (not all(isfinite(v) for v in (n, e, confidence))
            or not 0 <= e <= n or not 0 < confidence < 1):
        raise ValueError("Require 0 <= errors <= weight and 0 < confidence < 1.")
    if n == 0:
        return 0.0
    zero_errors = -n * expm1(log(confidence) / n)
    if e == 0:
        return zero_errors
    if e == n:
        return n
    if e < 1:
        anchor = min(1.0, n)
        upper = errores_estimados(n, anchor, confidence)
        return zero_errors + e / anchor * (upper - zero_errors)
    if e + 0.5 >= n:
        return e + 0.67 * (n - e)
    levels = (0.0, 0.001, 0.005, 0.01, 0.05, 0.10, 0.20, 0.40, 1.0)
    deviations = (4.0, 3.09, 2.58, 2.33, 1.65, 1.28, 0.84, 0.25, 0.0)
    index = next(i for i in range(1, len(levels)) if confidence <= levels[i])
    fraction = (confidence - levels[index - 1]) / (levels[index] - levels[index - 1])
    z = deviations[index - 1] + fraction * (deviations[index] - deviations[index - 1])
    f = (e + 0.5) / n
    upper = (f + z * z / (2 * n) + z * sqrt(f * (1 - f) / n + z * z / (4 * n * n)))
    return n * upper / (1 + z * z / n)


def U_CF(E, N, CF=0.25):
    """Devuelve una aproximación a la tasa U_CF(E,N), no al número de errores.
    Fuente conceptual: POD, libro pp. 41–42; PDF pp. 5–6, §§4.2–4.3.
    Conserva la notación y el orden E,N del libro; usa la rutina aproximada propuesta.
    Para N=0 devuelve cero por convención local, no por una definición del PDF.
    """
    estimated = errores_estimados(N, E, CF)
    return estimated / N if N else 0.0


def estadisticas_hoja(rows, weights=None, confidence=0.25):
    """Describe una hoja mayoritaria hipotética, sin crear ni modificar nodos.
    Fuente conceptual: POD, libro pp. 37–42; PDF pp. 1–6, §§4.2–4.3.
    El nombre y el diccionario son propios; usa frecuencias y errores ponderados.
    """
    counts = frecuencias_clase(rows, weights)
    total = sum(counts.values())
    prediction = max(counts, key=counts.get, default=None)
    errors = total - max(counts.values(), default=0.0)
    return {"prediction": prediction, "class_counts": counts, "total_weight": total,
            "observed_errors": errors,
            "estimated_errors": errores_estimados(total, errors, confidence)}


def elegir_poda(subtree_errors, leaf_errors, raised_branch_errors=None):
    """Compara errores del subárbol, la hoja y una rama elevada sobre los mismos casos.
    Fuente conceptual: POD, libro pp. 37–39; PDF pp. 1–3, §4.2.
    El nombre y las etiquetas keep/leaf/raise son propios. No recorre ni modifica el árbol.
    Ante igualdad con el subárbol actual lo conserva; entre mejoras iguales prefiere la hoja.
    """
    candidates = [("keep", subtree_errors), ("leaf", leaf_errors)]
    if raised_branch_errors is not None:
        candidates.append(("raise", raised_branch_errors))
    return min(candidates, key=lambda candidate: candidate[1])[0]


In [ ]:
class NodoEnriquecido:
    """Estado ponderado propio; NUL, libro pp. 29–32 / PDF pp. 3–6."""
    def __init__(self, filas, pesos):
        self.class_counts = frecuencias_clase(filas, pesos)
        self.total_weight = sum(self.class_counts.values())
        self.class_probabilities = probabilidades_clase(filas, pesos)
        self.prediction = max(self.class_counts, key=self.class_counts.get, default=None)
        self.attribute = None
        self.threshold = None
        self.children = {}
        self.branch_probabilities = {}


def _seleccionar_prueba_enriquecida(filas, atributos, pesos, continuos, referencias, minimo, criterio):
    """Enumera pruebas y elige la mayor puntuación; conserva el primer empate.
    NUL, libro pp. 28–31 / PDF pp. 2–5; CON, libro pp. 25–26 / PDF pp. 1–2.
    Cada ganancia usa sus casos conocidos; no se resta una impureza a un p0 global.
    Sólo considera ganancias positivas; el crecimiento comprueba después min_gain.
    """
    mejor = None
    for atributo in atributos:
        if atributo in continuos:
            candidatos = evaluar_umbrales_continuos(
                filas, atributo, pesos, referencias[atributo], minimo)
        else:
            prueba = evaluar_prueba(filas, atributo, pesos)
            candidatos = [] if prueba is None else [prueba]
        for prueba in candidatos:
            if prueba['gain'] > 0 and (mejor is None or prueba[criterio] > mejor[1][criterio]):
                mejor = atributo, prueba
    return mejor


def _poblaciones_enriquecidas(nodo, filas, pesos):
    """Redirige la población completa; NUL pp. 29–32 / PDF 3–6; POD pp. 37–39 / PDF 1–3.
    Las categorías nuevas quedan en el respaldo; los nulos se reparten sin inventar ramas.
    """
    utilizables, ponderaciones, respaldo, pesos_respaldo = [], [], [], []
    for fila, peso in zip(filas, pesos):
        valor = fila[nodo.attribute]
        desconocido = es_desconocido(valor)
        rama = None if desconocido else (valor if nodo.threshold is None else valor <= nodo.threshold)
        if desconocido or rama in nodo.children:
            utilizables.append(fila)
            ponderaciones.append(peso)
        else:
            respaldo.append(fila)
            pesos_respaldo.append(peso)
    grupos, probabilidades = particionar_con_desconocidos(
        utilizables, nodo.attribute, ponderaciones, nodo.threshold)
    return grupos, probabilidades, respaldo, pesos_respaldo


def _error_respaldo_enriquecido(nodo, filas, pesos, confianza):
    """Estima el respaldo propio con la misma aproximación; POD pp. 40–42 / PDF 4–6."""
    errores = sum(p for f, p in zip(filas, pesos) if f[-1] != nodo.prediction)
    return errores_estimados(sum(pesos), errores, confianza)


def _recalibrar_enriquecido(plantilla, filas, pesos, confianza):
    """Copia y reestima sobre TODOS los casos; POD, libro pp. 37–43 / PDF pp. 1–7.
    No conserva estadísticas del antiguo subconjunto de una rama elevada.
    """
    nodo = NodoEnriquecido(filas, pesos)
    hoja = estadisticas_hoja(filas, pesos, confianza)
    if plantilla.attribute is None or nodo.total_weight == 0:
        return nodo, hoja['estimated_errors']
    grupos, probabilidades, respaldo, pesos_respaldo = _poblaciones_enriquecidas(plantilla, filas, pesos)
    if not grupos:
        return nodo, hoja['estimated_errors']
    nodo.attribute, nodo.threshold = plantilla.attribute, plantilla.threshold
    nodo.branch_probabilities = probabilidades
    error = _error_respaldo_enriquecido(nodo, respaldo, pesos_respaldo, confianza)
    for rama, (grupo, ponderaciones) in grupos.items():
        hijo, estimacion = _recalibrar_enriquecido(
            plantilla.children[rama], grupo, ponderaciones, confianza)
        nodo.children[rama] = hijo
        error += estimacion
    return nodo, error


def _podar_enriquecido(plantilla, filas, pesos, confianza):
    """Poda ascendente real: hoja, conservación o elevación; POD pp. 37–43 / PDF 1–7.
    Compara las tres opciones en la misma población, con la aproximación aprobada.
    """
    nodo, error = _recalibrar_enriquecido(plantilla, filas, pesos, confianza)
    if nodo.attribute is None:
        return nodo, error
    grupos, _, respaldo, pesos_respaldo = _poblaciones_enriquecidas(nodo, filas, pesos)
    error = _error_respaldo_enriquecido(nodo, respaldo, pesos_respaldo, confianza)
    for rama, (grupo, ponderaciones) in grupos.items():
        nodo.children[rama], estimacion = _podar_enriquecido(
            nodo.children[rama], grupo, ponderaciones, confianza)
        error += estimacion
    hoja = estadisticas_hoja(filas, pesos, confianza)
    mas_usada = max(nodo.children.values(), key=lambda hijo: hijo.total_weight)
    elevada, error_elevada = _recalibrar_enriquecido(mas_usada, filas, pesos, confianza)
    decision = elegir_poda(error, hoja['estimated_errors'], error_elevada)
    if decision == 'leaf':
        return NodoEnriquecido(filas, pesos), hoja['estimated_errors']
    if decision == 'raise':
        return elevada, error_elevada
    return nodo, error


def predecir_probabilidades_enriquecido(arbol, fila):
    """Combina ramas ante nulos; NUL, libro pp. 30–33 / PDF pp. 4–7.
    Una categoría nueva usa la distribución del nodo como respaldo determinista propio.
    """
    if arbol.attribute is None:
        return dict(arbol.class_probabilities)
    valor = fila[arbol.attribute]
    if es_desconocido(valor):
        distribuciones = {rama: predecir_probabilidades_enriquecido(hijo, fila)
                          for rama, hijo in arbol.children.items()}
        combinada = combinar_probabilidades_clase(distribuciones, arbol.branch_probabilities)
        total = sum(combinada.values())
        return {clase: p / total for clase, p in combinada.items()} if total else {}
    rama = valor if arbol.threshold is None else valor <= arbol.threshold
    if rama not in arbol.children:
        return dict(arbol.class_probabilities)
    return predecir_probabilidades_enriquecido(arbol.children[rama], fila)


def predecir_enriquecido(arbol, fila):
    """Elige la clase más probable; empate por orden del nodo, convención propia.
    Fuente conceptual de la distribución: NUL, libro pp. 30–33 / PDF pp. 4–7.
    """
    probabilidades = predecir_probabilidades_enriquecido(arbol, fila)
    return max(arbol.class_counts, key=lambda clase: probabilidades.get(clase, 0), default=None)


In [ ]:
def _crecer_enriquecido(filas, atributos, pesos, continuos, referencias, min_gain, minimo,
                        max_depth, min_samples_split, depth, peso_raiz, criterio):
    """Integra ganancia y cortes: NUL pp. 28–31 / PDF 2–5; CON pp. 25–26 / PDF 1–2."""
    nodo = NodoEnriquecido(filas, pesos)
    if nodo.total_weight == 0 or len(nodo.class_counts) == 1:  # L1–L2: peso vacío o clase pura.
        return nodo
    elif not atributos:  # L3–L4: hoja mayoritaria sin atributos disponibles.
        return nodo
    elif max_depth is not None and depth >= max_depth:  # La raíz tiene profundidad cero.
        return nodo
    elif len(filas) < min_samples_split:  # Cuenta filas; la igualdad permite dividir.
        return nodo
    elif info(filas, pesos) == 0:  # Conserva la parada ponderada por entropía cero.
        return nodo
    else:
        mejor = _seleccionar_prueba_enriquecida(
            filas, atributos, pesos, continuos, referencias, minimo, criterio)  # L7–L11.
        if mejor is None:  # Sin dos ramas conocidas utilizables, no hay progreso.
            return nodo
        Ag, prueba = mejor
        ganancia_global = sum(pesos) / peso_raiz * prueba['gain']
        if not (prueba['gain'] > 0 and ganancia_global >= min_gain):  # L12–L13.
            return nodo
        else:
            nodo.attribute, nodo.threshold = Ag, prueba['threshold']  # L15.
            grupos, nodo.branch_probabilities = particionar_con_desconocidos(
                filas, Ag, pesos, nodo.threshold)  # L16: también propaga pesos fraccionarios.
            remaining = [a for a in atributos if a != Ag or a in continuos]  # L20.
            for rama, (group, pesos_grupo) in grupos.items():  # L17–L18: grupos con peso.
                nodo.children[rama] = _crecer_enriquecido(
                    group, remaining, pesos_grupo, continuos, referencias, min_gain, minimo,
                    max_depth=max_depth, min_samples_split=min_samples_split,
                    depth=depth + 1, peso_raiz=peso_raiz, criterio=criterio)  # L19–L20.
            return nodo  # L21–L24: la poda se aplica fuera del crecimiento.


def entrenar_arbol_enriquecido(filas, atributos, continuos=(), pesos=None,
                               min_gain=1e-12, min_peso_rama=2.0, confianza=0.25, podar=True,
                               max_depth=None, min_samples_split=2, criterio='gain'):
    """Entrena sin alterar la base ni las entradas; clase en la última columna.
    continuos declara índices; min_peso_rama exige peso conocido por lado continuo.
    max_depth=None no limita; cero deja la raíz como hoja. min_samples_split cuenta filas.
    Exige gain > 0 y (peso del nodo / peso original) * gain >= min_gain.
    Conserva el ajuste por desconocidos; podar=False permite inspeccionar el árbol crecido.
    Fuentes conceptuales: CON pp. 25–26 / PDF 1–2; NUL pp. 27–33 / PDF 1–7;
    POD pp. 37–43 / PDF 1–7. APIs propias y estimador aproximado, no código del libro.
    """
    if criterio not in ('gain', 'gain_ratio'):
        raise ValueError('El criterio debe ser gain o gain_ratio.')
    pesos = pesos_casos(filas, pesos)
    atributos, continuos = list(atributos), set(continuos)
    referencias = {a: [fila[a] for fila in filas] for a in continuos}
    arbol = _crecer_enriquecido(
        filas, atributos, pesos, continuos, referencias, min_gain, min_peso_rama,
        max_depth=max_depth, min_samples_split=min_samples_split, depth=0, peso_raiz=sum(pesos), criterio=criterio)
    return _podar_enriquecido(arbol, filas, pesos, confianza)[0] if podar else arbol


## 3. Visualización del árbol

Se adapta la visualización del árbol base a los nodos enriquecidos.
En las hojas se muestra el peso de la clase predicha y el peso total.
El gráfico muestra hasta profundidad 3, contando la raíz como 0.
Los puntos suspensivos indican ramas que continúan.


In [ ]:
def tree_layout(tree, max_display_depth=3):
    """Space leaves evenly and center each parent over its children."""
    nodes, edges = [], []
    next_leaf = 0

    def visit(node, depth):
        nonlocal next_leaf
        children = [(branch, visit(child, depth + 1))
                    for branch, child in node.children.items()] if depth < max_display_depth else []
        if children:
            x = (children[0][1][0] + children[-1][1][0]) / 2
        else:
            x = next_leaf
            next_leaf += 1
        position = (x, -depth)
        nodes.append((node, position))
        for branch, child_position in children:
            label = str(branch)
            if node.threshold is not None:
                label = f"{'<=' if branch else '>'} {node.threshold:g}"
            edges.append((position, child_position, label))
        return position

    visit(tree, 0)
    return nodes, edges


def plot_tree(tree, columns, max_display_depth=3):
    """Display the custom tree without changing it or installing dependencies."""
    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError as error:
        if error.name != 'matplotlib':
            raise
        print('Tree plot unavailable: matplotlib is not installed; see the text tree above.')
        return None

    nodes, edges = tree_layout(tree, max_display_depth)
    width = max(position[0] for _, position in nodes) + 1
    depth = -min(position[1] for _, position in nodes)
    fig, ax = plt.subplots(figsize=(max(6, min(24, 1.7 * width)), max(3, 1.8 * (depth + 1))))
    for start, end, label in edges:
        ax.plot([start[0], end[0]], [start[1], end[1]], color='0.5', zorder=1)
        ax.text((start[0] + end[0]) / 2, (start[1] + end[1]) / 2, label,
                ha='center', va='center', parse_math=False,
                bbox=dict(facecolor='white', edgecolor='none'), zorder=2)
    for node, (x, y) in nodes:
        leaf = node.attribute is None
        label = f'Class: {node.prediction}' if leaf else str(columns[node.attribute])
        if leaf:
            total = node.total_weight
            matching = node.class_counts.get(node.prediction, 0)
            label += f'\nPeso de clase / total: {matching:.1f} / {total:.1f}'
        if not leaf and -y == max_display_depth:
            label += '\n…'
        ax.text(x, y, label, ha='center', va='center', parse_math=False,
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#d9ead3' if leaf else '#cfe2f3',
                          edgecolor='0.4'), zorder=3)
    ax.set_xlim(-0.75, width - 0.25)
    ax.set_ylim(-depth - 0.6, 0.6)
    ax.set_axis_off()
    fig.tight_layout()
    plt.show()
    return fig, ax


## 4. Comparación

Cada paciente aparece una sola vez en prueba. Las métricas y matrices
de confusión se calculan con las predicciones de prueba reunidas de
las cinco particiones. Los tamaños del árbol se promedian entre las
cinco versiones entrenadas.

Para Random Forest, las columnas numéricas faltantes se rellenan con
la mediana del entrenamiento y las categóricas con su moda. Después,
solo las categóricas se codifican con One-Hot Encoding. El árbol propio
usa su manejo ponderado de valores faltantes.


In [ ]:
def crear_random_forest(n_estimators, max_depth, min_samples_split):
    transformacion_numerica = Pipeline([
        ("faltantes", SimpleImputer(strategy="median")),
    ])
    transformacion_categorica = Pipeline([
        ("faltantes", SimpleImputer(strategy="most_frequent")),
        ("categorias", OneHotEncoder(handle_unknown="ignore")),
    ])
    preparacion = ColumnTransformer([
        ("numericas", transformacion_numerica, columnas_numericas),
        ("categoricas", transformacion_categorica, columnas_categoricas),
    ])
    return Pipeline([
        ("preparacion", preparacion),
        ("forest", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1,
        )),
    ])


def dimensiones_arbol(nodo):
    if not nodo.children:
        return 1, 0
    hijos = [dimensiones_arbol(hijo) for hijo in nodo.children.values()]
    return 1 + sum(cantidad for cantidad, _ in hijos), 1 + max(profundidad for _, profundidad in hijos)


def comparar_modelos(opciones_dt, opciones_rf):
    particiones = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    reales = y.to_numpy()
    predicciones_dt = np.empty(len(X), dtype=int)
    predicciones_rf = np.empty(len(X), dtype=int)
    estadisticas = []
    arbol_ejemplo = None

    for numero, (entrenamiento, prueba) in enumerate(particiones.split(X, y), start=1):
        X_entrenamiento = X.iloc[entrenamiento]
        X_prueba = X.iloc[prueba]
        y_entrenamiento = y.iloc[entrenamiento]
        y_prueba = y.iloc[prueba]

        filas = [atributos_fila + [clase] for atributos_fila, clase in
                 zip(X_entrenamiento.to_numpy().tolist(), y_entrenamiento.tolist())]
        arbol = entrenar_arbol_enriquecido(
            filas, atributos, continuos=indices_continuos, **opciones_dt
        )
        bosque = crear_random_forest(**opciones_rf)
        bosque.fit(X_entrenamiento, y_entrenamiento)

        pred_dt = [predecir_enriquecido(arbol, fila)
                   for fila in X_prueba.to_numpy().tolist()]
        pred_rf = bosque.predict(X_prueba)
        predicciones_dt[prueba] = pred_dt
        predicciones_rf[prueba] = pred_rf
        nodos, profundidad = dimensiones_arbol(arbol)
        estadisticas.append({
            "Partición": numero,
            "Filas de prueba": len(prueba),
            "Accuracy DT": accuracy_score(y_prueba, pred_dt),
            "Accuracy RF": accuracy_score(y_prueba, pred_rf),
            "Nodos DT": nodos,
            "Profundidad DT": profundidad,
        })
        if arbol_ejemplo is None:
            arbol_ejemplo = arbol

    resumen = pd.DataFrame([
        {"Modelo": nombre,
         "Aciertos": int(np.sum(predicciones == reales)),
         "Casos": len(reales),
         "Accuracy": accuracy_score(reales, predicciones),
         "F1 macro": f1_score(reales, predicciones, average="macro", zero_division=0)}
        for nombre, predicciones in [
            ("Decision Tree", predicciones_dt),
            ("Random Forest", predicciones_rf),
        ]
    ]).set_index("Modelo")
    return {
        "resumen": resumen,
        "particiones": pd.DataFrame(estadisticas),
        "reales": reales,
        "predicciones_dt": predicciones_dt,
        "predicciones_rf": predicciones_rf,
        "arbol_ejemplo": arbol_ejemplo,
        "opciones_dt": dict(opciones_dt),
        "opciones_rf": dict(opciones_rf),
    }


def mostrar_resultados(resultado, mostrar_arbol=True):
    print(f"Resultados sobre las {len(resultado['reales'])} predicciones de prueba:")
    display(resultado["resumen"].style.format({"Accuracy": "{:.2%}", "F1 macro": "{:.2%}"}))
    print("Resultados por partición:")
    display(resultado["particiones"].style.format({"Accuracy DT": "{:.2%}", "Accuracy RF": "{:.2%}"}))
    print(f"Decision Tree promedio: {resultado['particiones']['Nodos DT'].mean():.1f} nodos, "
          f"profundidad {resultado['particiones']['Profundidad DT'].mean():.1f}.")
    print("Configuración del Decision Tree:", resultado["opciones_dt"])
    print("Configuración del Random Forest:", resultado["opciones_rf"])

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    for axis, nombre, predicciones in [
        (axes[0], "Decision Tree", resultado["predicciones_dt"]),
        (axes[1], "Random Forest", resultado["predicciones_rf"]),
    ]:
        ConfusionMatrixDisplay.from_predictions(
            resultado["reales"], predicciones, labels=[0, 1],
            display_labels=["0", "1"], ax=axis, colorbar=False,
        )
        axis.set_title(nombre)
        axis.set_xlabel("Clase predicha")
        axis.set_ylabel("Clase real")
    fig.tight_layout()
    plt.show()

    if mostrar_arbol:
        print("Árbol de la primera partición, hasta profundidad 3:")
        plot_tree(resultado["arbol_ejemplo"], X.columns, max_display_depth=3)


## 5. Panel de comparación

Cada pulsación vuelve a entrenar los dos modelos con la configuración
elegida y las mismas cinco particiones. El cálculo puede tardar unos
momentos. La profundidad de la raíz es 0 para el Decision Tree.


In [ ]:
def crear_panel_comparacion():
    estilo = {"description_width": "initial"}
    controles = {
        "criterio": widgets.Dropdown(
            options=[("Information Gain", "gain"), ("Gain Ratio", "gain_ratio")],
            description="Criterio", style=estilo,
        ),
        "dt_sin_limite": widgets.Checkbox(value=True, description="Sin límite de profundidad"),
        "dt_profundidad": widgets.BoundedIntText(
            value=5, min=0, max=50, description="Profundidad máxima DT", style=estilo,
        ),
        "dt_min_filas": widgets.BoundedIntText(
            value=2, min=2, max=303, description="Mínimo de filas para dividir DT", style=estilo,
        ),
        "podar": widgets.Checkbox(value=True, description="Aplicar poda"),
        "min_gain": widgets.FloatText(value=1e-12, description="Ganancia mínima", style=estilo),
        "min_peso_rama": widgets.FloatText(value=2.0, description="Peso mínimo por lado", style=estilo),
        "confianza": widgets.FloatText(value=0.25, description="Confianza de poda", style=estilo),
        "n_estimators": widgets.BoundedIntText(
            value=100, min=1, max=1000, description="Cantidad de árboles RF", style=estilo,
        ),
        "rf_sin_limite": widgets.Checkbox(value=True, description="Sin límite de profundidad"),
        "rf_profundidad": widgets.BoundedIntText(
            value=5, min=1, max=50, description="Profundidad máxima RF", style=estilo,
        ),
        "rf_min_filas": widgets.BoundedIntText(
            value=2, min=2, max=303, description="Mínimo de filas para dividir RF", style=estilo,
        ),
        "mostrar_arbol": widgets.Checkbox(value=True, description="Mostrar árbol de ejemplo"),
    }
    for control in controles.values():
        control.layout.width = "440px"
        control.layout.max_width = "100%"
    avanzadas = widgets.Accordion(children=[
        widgets.VBox([controles["min_gain"], controles["min_peso_rama"], controles["confianza"]])
    ])
    avanzadas.set_title(0, "Opciones avanzadas del Decision Tree")
    avanzadas.selected_index = None
    boton = widgets.Button(description="Comparar modelos", icon="play", button_style="primary")
    salida = widgets.Output()
    panel = widgets.VBox([
        widgets.HTML("<h3>Decision Tree</h3>"),
        controles["criterio"], controles["dt_sin_limite"], controles["dt_profundidad"],
        controles["dt_min_filas"], controles["podar"], avanzadas,
        widgets.HTML("<h3>Random Forest</h3>"),
        controles["n_estimators"], controles["rf_sin_limite"],
        controles["rf_profundidad"], controles["rf_min_filas"],
        controles["mostrar_arbol"], boton, salida,
    ])
    panel.controles = controles
    panel.boton_comparar = boton
    panel.ultimo_resultado = None

    def sincronizar(_=None):
        controles["dt_profundidad"].disabled = controles["dt_sin_limite"].value
        controles["rf_profundidad"].disabled = controles["rf_sin_limite"].value
        controles["confianza"].disabled = not controles["podar"].value

    for nombre in ("dt_sin_limite", "rf_sin_limite", "podar"):
        controles[nombre].observe(sincronizar, names="value")
    sincronizar()

    def ejecutar(_):
        with salida:
            salida.clear_output(wait=True)
            boton.disabled = True
            try:
                ganancia = controles["min_gain"].value
                peso = controles["min_peso_rama"].value
                confianza = controles["confianza"].value
                if not np.isfinite(ganancia) or ganancia < 0:
                    raise ValueError("La ganancia mínima debe ser finita y mayor o igual a cero.")
                if not np.isfinite(peso) or peso <= 0:
                    raise ValueError("El peso mínimo por lado debe ser finito y mayor que cero.")
                if not np.isfinite(confianza) or not 0 < confianza < 1:
                    raise ValueError("La confianza de poda debe estar entre 0 y 1.")

                opciones_dt = {
                    "criterio": controles["criterio"].value,
                    "max_depth": None if controles["dt_sin_limite"].value else controles["dt_profundidad"].value,
                    "min_samples_split": controles["dt_min_filas"].value,
                    "podar": controles["podar"].value,
                    "min_gain": ganancia,
                    "min_peso_rama": peso,
                    "confianza": confianza,
                }
                opciones_rf = {
                    "n_estimators": controles["n_estimators"].value,
                    "max_depth": None if controles["rf_sin_limite"].value else controles["rf_profundidad"].value,
                    "min_samples_split": controles["rf_min_filas"].value,
                }
                print("Entrenando y evaluando las cinco particiones...")
                resultado = comparar_modelos(opciones_dt, opciones_rf)
                panel.ultimo_resultado = resultado
                salida.clear_output(wait=True)
                mostrar_resultados(resultado, mostrar_arbol=controles["mostrar_arbol"].value)
            except (ValueError, TypeError, OverflowError) as error:
                print(f"No se pudo completar la comparación: {error}")
            except Exception as error:
                import traceback
                print(f"No se pudo completar la comparación: {error}")
                traceback.print_exc()
            finally:
                boton.disabled = False

    boton.on_click(ejecutar)
    return panel


panel = crear_panel_comparacion()
display(panel)


## Referencias

- Janosi, A., Steinbrunn, W., Pfisterer, M. y Detrano, R. (1989).
  Heart Disease. UCI Machine Learning Repository.
  https://doi.org/10.24432/C52P4X
- Quinlan, J. R. C4.5: Programs for Machine Learning. Morgan Kaufmann, 1993.
  La implementación del equipo incorpora adaptaciones propias para el
  manejo de faltantes, los cortes y la estimación de errores de poda.
- Breiman, L. Random Forests. Machine Learning 45, 5–32 (2001).
- Scikit-learn. Documentación de RandomForestClassifier y preprocesamiento.
  https://scikit-learn.org/stable/
